# CAT 1: Uganda road traffic accidents (2011–2020)
**Godsgreat Nte | 161-761**
The analysis loads the annual summary directly from GitHub. Run after uploading the cleaned files.


In [ ]:
import csv, io, urllib.request
url = "https://raw.githubusercontent.com/ggee2400/cavendish-road-safety-warehouse/main/data/annual_summary.csv"
response = urllib.request.urlopen(url, timeout=30)
data = list(csv.DictReader(io.StringIO(response.read().decode("utf-8-sig"))))
print("Imported from GitHub:", url)
print("Records:", len(data), "Years:", data[0]["Year"], "to", data[-1]["Year"])


In [ ]:
from math import sqrt
variables = ["Population_Behavior", "Road_Environment", "Motor_Vehicle_Condition", "Total_Road_Traffic_Accidents"]
def pearson_pairs(a, b):
    pairs = [(float(row[a]), float(row[b])) for row in data if row[a].strip() and row[b].strip()]
    mx = sum(x for x,y in pairs) / len(pairs)
    my = sum(y for x,y in pairs) / len(pairs)
    denominator = sqrt(sum((x-mx)**2 for x,y in pairs) * sum((y-my)**2 for x,y in pairs))
    return (sum((x-mx)*(y-my) for x,y in pairs)/denominator, len(pairs))
matrix = [[pearson_pairs(a,b)[0] for b in variables] for a in variables]
counts = [[pearson_pairs(a,b)[1] for b in variables] for a in variables]
for var, row in zip(variables,matrix): print(var, [round(r,3) for r in row])


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9.5, 7))
im = ax.imshow(matrix, cmap="RdBu_r", vmin=-1, vmax=1)
labels = ["Population\nbehaviour", "Road\nenvironment", "Vehicle\ncondition", "Total road\ntraffic accidents"]
ax.set_xticks(range(4), labels)
ax.set_yticks(range(4), labels)
for i in range(4):
    for j in range(4):
        r = matrix[i][j]
        ax.text(j, i, f"{r:+.3f}\n(n={counts[i][j]})", ha="center", va="center", color="white" if abs(r)>0.65 else "black", fontweight="bold")
ax.set_title("Pearson correlations, annual road crash counts (2011–2020)")
fig.colorbar(im, ax=ax, label="Pearson r")
fig.tight_layout()
fig.savefig("CAT1_GitHub_import_heatmap.png", dpi=220)
plt.show()


**Interpretation:** Population-behaviour counts correlate strongly with total crashes, but the predictors form part of that total. Correlation does not establish causation. Vehicle-related figures lack full records for 2011 and 2016 (n = 8 for relevant comparisons). The original source has documented internal discrepancies; see `data/data_quality_audit.csv`.
